### Weather + UK Energy Gap Models
Single-horizon (1-step) and Multi-horizon (1-12h) forecasting with holdout test evaluation.

In [46]:
%pip install skl2onnx onnx onnxruntime joblib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

In [48]:
DEMAND_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/uk/uk-data/demand.csv"
GEN_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/uk/uk-data/generation.csv"
WEATHER_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/weather-data/country_dfs/united_kingdom_london_2016-02-21_2026-02-21_hourly.csv"

# Holdout test window
TEST_START = pd.Timestamp("2026-02-09 00:00:00")
TEST_END   = pd.Timestamp("2026-02-16 00:00:00")

# Output directory for saved models
MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [49]:
uk_demand = pd.read_csv(DEMAND_PATH)
uk_generation = pd.read_csv(GEN_PATH)
uk_weather = pd.read_csv(WEATHER_PATH)

print(uk_weather.columns)
print(uk_generation.head())
print(uk_demand.head())

Index(['country', 'country_code', 'location', 'time', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
       'wind_direction_10m', 'surface_pressure', 'cloud_cover',
       'latitude_used', 'longitude_used', 'timezone'],
      dtype='object')
  Dataset           PublishTime             StartTime SettlementDate  \
0  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
1  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
2  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
3  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
4  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   

   SettlementPeriod FuelType  Generation  
0                47     CCGT        8648  
1                47     COAL        3973  
2                47    INTEW         282  
3                47    INTFR        1496  
4                47   INTIRL         186  
  RecordType             Start

In [50]:
print("Demand time range:")
print(f"  Start: {uk_demand['StartTime'].min()}")
print(f"  End:   {uk_demand['StartTime'].max()}")

Demand time range:
  Start: 2016-02-29T23:55:00Z
  End:   2026-02-21T13:00:00Z


### Data Merging and Preprocessing

In [51]:
# Merge Demand + Generation
gen_agg = (
    uk_generation
    .groupby("StartTime")["Generation"]
    .sum()
    .reset_index()
)

df = uk_demand.merge(gen_agg, on="StartTime", how="inner")
df = df.rename(columns={"Generation": "Total_Generation"})
df["Gap"] = df["Total_Generation"] - df["Demand"]

df["StartTime"] = pd.to_datetime(df["StartTime"])
df = df.sort_values("StartTime")

# Prepare weather + align datetime
uk_weather = uk_weather.rename(columns={"time": "StartTime"})
uk_weather["StartTime"] = pd.to_datetime(uk_weather["StartTime"])

# Resample demand/gen/gap to hourly (mean)
cols = ["Demand", "Total_Generation", "Gap"]
df_hourly = (
    df.set_index("StartTime")[cols]
      .resample("H")
      .mean()
      .reset_index()
)

# Normalize timezone handling
df_hourly["StartTime"] = pd.to_datetime(df_hourly["StartTime"], utc=True).dt.tz_convert(None)
uk_weather["StartTime"] = pd.to_datetime(uk_weather["StartTime"], utc=True).dt.tz_convert(None)

# Merge hourly df with weather
df_final = df_hourly.merge(uk_weather, on="StartTime", how="left")
df_final = df_final.dropna().reset_index(drop=True)
print(f"Final merged shape: {df_final.shape}")

Final merged shape: (86934, 17)


### Feature Engineering

In [52]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["StartTime"].dt.hour
    df["day_of_week"] = df["StartTime"].dt.dayofweek
    df["month"] = df["StartTime"].dt.month
    df["day_of_year"] = df["StartTime"].dt.dayofyear
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_lag_rolling_features(
    df: pd.DataFrame,
    target_col: str = "Gap",
    extra_series_cols=("Demand", "Total_Generation"),
    lags=(1, 2, 3, 6, 12, 24, 48, 168),
    rolls=(3, 6, 24, 168)
) -> pd.DataFrame:
    df = df.copy()

    for L in lags:
        df[f"{target_col}_lag_{L}"] = df[target_col].shift(L)

    shifted = df[target_col].shift(1)
    for W in rolls:
        df[f"{target_col}_roll_mean_{W}"] = shifted.rolling(W).mean()
        df[f"{target_col}_roll_std_{W}"] = shifted.rolling(W).std()
        df[f"{target_col}_roll_min_{W}"] = shifted.rolling(W).min()
        df[f"{target_col}_roll_max_{W}"] = shifted.rolling(W).max()

    for col in extra_series_cols:
        for L in lags:
            df[f"{col}_lag_{L}"] = df[col].shift(L)
        shifted_col = df[col].shift(1)
        for W in rolls:
            df[f"{col}_roll_mean_{W}"] = shifted_col.rolling(W).mean()
            df[f"{col}_roll_std_{W}"] = shifted_col.rolling(W).std()

    df[f"{target_col}_diff_1"] = df[target_col].diff(1)
    df[f"{target_col}_diff_24"] = df[target_col].diff(24)

    if "temperature_2m" in df.columns:
        df["temp_diff_1"] = df["temperature_2m"].diff(1)
        df["temp_diff_24"] = df["temperature_2m"].diff(24)

    return df


def make_supervised(df: pd.DataFrame, horizon: int = 1, target_col: str = "Gap") -> pd.DataFrame:
    df = df.copy()
    df["y"] = df[target_col].shift(-horizon)
    df = df.dropna(subset=["y"])
    return df


def rmse(y_true, y_pred) -> float:
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred, eps: float = 1e-6) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100


WEATHER_COLS = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure", "cloud_cover",
    "latitude_used", "longitude_used"
]

TIME_COLS_CYCLIC = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "day_of_year", "is_weekend"
]

---
## PART 1 -- Single-Horizon Model (1-step ahead)

In [53]:
HORIZON = 1

df_feat = add_time_features(df_final)
df_feat = add_lag_rolling_features(df_feat)
df_feat = make_supervised(df_feat, horizon=HORIZON, target_col="Gap")
df_feat = df_feat.dropna().reset_index(drop=True)

lag_roll_cols_single = [
    c for c in df_feat.columns
    if ("_lag_" in c) or ("_roll_" in c) or ("_diff_" in c)
]

feature_cols_single = WEATHER_COLS + TIME_COLS_CYCLIC + lag_roll_cols_single
feature_cols_single = [c for c in feature_cols_single if c in df_feat.columns]

X_single = df_feat[feature_cols_single].reset_index(drop=True)
y_single = df_feat["y"].reset_index(drop=True)
times_single = df_feat["StartTime"].reset_index(drop=True)

print(f"[Single-Horizon] X: {X_single.shape}, y: {y_single.shape}")

[Single-Horizon] X: (86765, 77), y: (86765,)


#### Single-Horizon: Holdout Test (Feb 9 -- Feb 16)

In [54]:
# Split using TEST_START / TEST_END
test_end_for_X = TEST_END - pd.Timedelta(hours=HORIZON)

train_mask_s = times_single < TEST_START
test_mask_s  = (times_single >= TEST_START) & (times_single < test_end_for_X)

X_train_s, X_test_s = X_single.loc[train_mask_s], X_single.loc[test_mask_s]
y_train_s, y_test_s = y_single.loc[train_mask_s], y_single.loc[test_mask_s]

print("Single-horizon holdout split:")
print(f"  TEST_START: {TEST_START}")
print(f"  TEST_END:   {TEST_END}")
print(f"  Train: {times_single.loc[train_mask_s].min()} -> {times_single.loc[train_mask_s].max()} | n={int(train_mask_s.sum())}")
print(f"  Test:  {times_single.loc[test_mask_s].min()} -> {times_single.loc[test_mask_s].max()} | n={int(test_mask_s.sum())}")

model_single = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("reg", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

model_single.fit(X_train_s, y_train_s)
preds_s = model_single.predict(X_test_s)

print(f"\n[Single-Horizon] Holdout Test Results ({TEST_START.date()} to {TEST_END.date()}):")
print(f"  MAE:    {mean_absolute_error(y_test_s, preds_s):.2f}")
print(f"  RMSE:   {rmse(y_test_s, preds_s):.2f}")
print(f"  R2:     {r2_score(y_test_s, preds_s):.4f}")
print(f"  MAPE %: {mape(y_test_s.values, preds_s):.2f}")

Single-horizon holdout split:
  TEST_START: 2026-02-09 00:00:00
  TEST_END:   2026-02-16 00:00:00
  Train: 2016-03-08 00:00:00 -> 2026-02-08 23:00:00 | n=86598
  Test:  2026-02-09 00:00:00 -> 2026-02-15 22:00:00 | n=167

[Single-Horizon] Holdout Test Results (2026-02-09 to 2026-02-16):
  MAE:    430.19
  RMSE:   533.48
  R2:     0.8974
  MAPE %: 30.12


#### Single-Horizon: TimeSeries Cross-Validation

In [55]:
TEST_SIZE = 30 * 24
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

fold_results_s = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_single), start=1):
    Xtr, Xte = X_single.iloc[train_idx], X_single.iloc[test_idx]
    ytr, yte = y_single.iloc[train_idx], y_single.iloc[test_idx]

    model_single.fit(Xtr, ytr)
    p = model_single.predict(Xte)

    fold_results_s.append({
        "fold":   fold,
        "MAE":    mean_absolute_error(yte, p),
        "RMSE":   rmse(yte, p),
        "R2":     r2_score(yte, p),
        "MAPE_%": mape(yte.values, p)
    })

results_single_cv = pd.DataFrame(fold_results_s)
print("[Single-Horizon] CV Results:")
print(results_single_cv)
print(f"\nMean:\n{results_single_cv[['MAE', 'RMSE', 'R2', 'MAPE_%']].mean()}")
print(f"\nStd:\n{results_single_cv[['MAE', 'RMSE', 'R2', 'MAPE_%']].std()}")

[Single-Horizon] CV Results:
   fold         MAE        RMSE        R2      MAPE_%
0     1  338.343708  465.627362  0.792742   33.220203
1     2  362.799197  484.210166  0.860285  105.408979
2     3  431.189660  578.476135  0.840256   40.837168
3     4  359.875776  505.851135  0.772382   60.029217
4     5  392.249841  530.623787  0.896717   48.588957

Mean:
MAE       376.891636
RMSE      512.957717
R2          0.832477
MAPE_%     57.616905
dtype: float64

Std:
MAE       35.910406
RMSE      43.937003
R2         0.050376
MAPE_%    28.496334
dtype: float64


#### Single-Horizon: Refit on train (up to TEST_START), save ONNX + weights

In [56]:
# Refit on training data (everything before TEST_START) for the final saved model
print("Refitting single-horizon model on train data (up to TEST_START)...")
model_single.fit(X_train_s, y_train_s)

# Save sklearn weights (joblib)
joblib_path_single = os.path.join(MODEL_DIR, "gap_single_horizon.joblib")
joblib.dump(model_single, joblib_path_single)
print(f"Saved sklearn weights: {joblib_path_single}")

# Save ONNX
initial_type_single = [("float_input", FloatTensorType([None, X_single.shape[1]]))]
onnx_single = convert_sklearn(model_single, initial_types=initial_type_single, target_opset=17)

onnx_path_single = os.path.join(MODEL_DIR, "gap_single_horizon.onnx")
with open(onnx_path_single, "wb") as f:
    f.write(onnx_single.SerializeToString())
print(f"Saved ONNX model: {onnx_path_single}")

# Save feature columns list for inference
joblib.dump(feature_cols_single, os.path.join(MODEL_DIR, "feature_cols_single.joblib"))

# Sanity check ONNX
sess_single = rt.InferenceSession(onnx_path_single)
sample = X_test_s.iloc[:3].values.astype(np.float32)
preds_onnx = sess_single.run(None, {sess_single.get_inputs()[0].name: sample})
print(f"ONNX sanity check output shape: {np.array(preds_onnx[0]).shape}")

# Verify ONNX matches sklearn
preds_sk = model_single.predict(X_test_s.iloc[:3])
print(f"Max difference sklearn vs ONNX: {np.max(np.abs(preds_sk - preds_onnx[0].flatten())):.6f}")

Refitting single-horizon model on train data (up to TEST_START)...
Saved sklearn weights: saved_models/gap_single_horizon.joblib
Saved ONNX model: saved_models/gap_single_horizon.onnx
ONNX sanity check output shape: (3, 1)
Max difference sklearn vs ONNX: 157.080156


In [61]:
# Save single-horizon test set as parquet and npy
test_df_single = X_test_s.copy()
test_df_single["StartTime"] = times_single.loc[test_mask_s].values
test_df_single["y_true"] = y_test_s.values
test_df_single["y_pred"] = preds_s
test_df_single.to_parquet(os.path.join(MODEL_DIR, "test_single_horizon.parquet"), index=False)
np.save(os.path.join(MODEL_DIR, "test_single_horizon_X.npy"), X_test_s.values)
np.save(os.path.join(MODEL_DIR, "test_single_horizon_y.npy"), y_test_s.values)
print(f"Saved test_single_horizon.parquet  shape: {test_df_single.shape}")
print(f"Saved test_single_horizon_X.npy    shape: {X_test_s.values.shape}")
print(f"Saved test_single_horizon_y.npy    shape: {y_test_s.values.shape}")


Saved test_single_horizon.parquet  shape: (167, 80)
Saved test_single_horizon_X.npy    shape: (167, 77)
Saved test_single_horizon_y.npy    shape: (167,)


---
## PART 2 -- Multi-Horizon Model 

In [58]:
HORIZONS = range(1, 13)

df_model = df_final.sort_values("StartTime").reset_index(drop=True).copy()
df_model = add_time_features(df_model)
independent_cols = df_model.columns.tolist()

# Create future targets
for h in HORIZONS:
    df_model[f"Gap_t_plus_{h}h"] = df_model["Gap"].shift(-h)

dependent_cols = [c for c in df_model.columns if c not in independent_cols]
print(f"[Multi-Horizon] Target columns ({len(dependent_cols)}): {dependent_cols}")


def add_lags(df: pd.DataFrame, col: str, lags=(1, 2, 3, 6, 12, 24, 48, 168)) -> pd.DataFrame:
    df = df.copy()
    for L in lags:
        df[f"{col}_lag_{L}"] = df[col].shift(L)
    return df


def add_rolls(df: pd.DataFrame, col: str, windows=(3, 6, 24, 168)) -> pd.DataFrame:
    df = df.copy()
    s = df[col].shift(1)
    for W in windows:
        df[f"{col}_roll_mean_{W}"] = s.rolling(W).mean()
        df[f"{col}_roll_std_{W}"]  = s.rolling(W).std()
    return df


df_model = add_lags(df_model, "Gap")
df_model = add_rolls(df_model, "Gap")
df_model = add_lags(df_model, "Demand")
df_model = add_lags(df_model, "Total_Generation")

df_model["Gap_diff_1"]  = df_model["Gap"].diff(1)
df_model["Gap_diff_24"] = df_model["Gap"].diff(24)
if "temperature_2m" in df_model.columns:
    df_model["temp_diff_1"]  = df_model["temperature_2m"].diff(1)
    df_model["temp_diff_24"] = df_model["temperature_2m"].diff(24)

lag_roll_cols_multi = [
    c for c in df_model.columns
    if ("_lag_" in c) or ("_roll_" in c) or ("_diff_" in c)
]

feature_cols_multi = WEATHER_COLS + TIME_COLS_CYCLIC + lag_roll_cols_multi
feature_cols_multi = [c for c in feature_cols_multi if c in df_model.columns]

# Drop NaNs
data = pd.concat([df_model[feature_cols_multi], df_model[dependent_cols], df_model[["StartTime"]]], axis=1)
data = data.dropna().reset_index(drop=True)

X_multi = data[feature_cols_multi]
Y_multi = data[dependent_cols]
times_multi = data["StartTime"]

print(f"[Multi-Horizon] X: {X_multi.shape}, Y: {Y_multi.shape}")

[Multi-Horizon] Target columns (12): ['Gap_t_plus_1h', 'Gap_t_plus_2h', 'Gap_t_plus_3h', 'Gap_t_plus_4h', 'Gap_t_plus_5h', 'Gap_t_plus_6h', 'Gap_t_plus_7h', 'Gap_t_plus_8h', 'Gap_t_plus_9h', 'Gap_t_plus_10h', 'Gap_t_plus_11h', 'Gap_t_plus_12h']
[Multi-Horizon] X: (86754, 53), Y: (86754, 12)


#### Multi-Horizon: Holdout Test (Feb 9 -- Feb 16)

In [59]:
# For multi-horizon, the latest target is Gap(t+12), so exclude feature rows
# where t+12 >= TEST_END
max_horizon = max(HORIZONS)
test_end_for_X_multi = TEST_END - pd.Timedelta(hours=max_horizon)

train_mask_m = times_multi < TEST_START
test_mask_m  = (times_multi >= TEST_START) & (times_multi < test_end_for_X_multi)

X_train_m, X_test_m = X_multi.loc[train_mask_m], X_multi.loc[test_mask_m]
Y_train_m, Y_test_m = Y_multi.loc[train_mask_m], Y_multi.loc[test_mask_m]

print("Multi-horizon holdout split:")
print(f"  TEST_START: {TEST_START}")
print(f"  TEST_END:   {TEST_END}")
print(f"  Train: {times_multi.loc[train_mask_m].min()} -> {times_multi.loc[train_mask_m].max()} | n={int(train_mask_m.sum())}")
print(f"  Test:  {times_multi.loc[test_mask_m].min()} -> {times_multi.loc[test_mask_m].max()} | n={int(test_mask_m.sum())}")

base_estimator = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=8,
    max_iter=500,
    random_state=42
)
model_multi = MultiOutputRegressor(base_estimator)

model_multi.fit(X_train_m, Y_train_m)
Y_pred_m = model_multi.predict(X_test_m)

print(f"\n[Multi-Horizon] Holdout Test Results ({TEST_START.date()} to {TEST_END.date()}):")
holdout_rows = []
for j, h in enumerate(HORIZONS):
    y_true_h = Y_test_m.iloc[:, j].values
    y_pred_h = Y_pred_m[:, j]
    holdout_rows.append({
        "horizon_h": h,
        "MAE":  mean_absolute_error(y_true_h, y_pred_h),
        "RMSE": rmse(y_true_h, y_pred_h),
        "R2":   r2_score(y_true_h, y_pred_h),
    })

holdout_multi = pd.DataFrame(holdout_rows).set_index("horizon_h")
print(holdout_multi)
print(f"\nOverall mean:\n{holdout_multi.mean()}")

Multi-horizon holdout split:
  TEST_START: 2026-02-09 00:00:00
  TEST_END:   2026-02-16 00:00:00
  Train: 2016-03-08 00:00:00 -> 2026-02-08 23:00:00 | n=86598
  Test:  2026-02-09 00:00:00 -> 2026-02-15 11:00:00 | n=156

[Multi-Horizon] Holdout Test Results (2026-02-09 to 2026-02-16):
                   MAE         RMSE        R2
horizon_h                                    
1           437.978559   544.907116  0.854731
2           778.524582   958.259363  0.571827
3          1025.336384  1239.343153  0.315090
4          1223.772265  1481.000600  0.064565
5          1381.543957  1653.765587 -0.108165
6          1476.270882  1749.690798 -0.186866
7          1416.526233  1749.325995 -0.154284
8          1516.018699  1837.356051 -0.244613
9          1574.104323  1874.035230 -0.263426
10         1666.121532  2005.465141 -0.423945
11         1653.107164  1989.524670 -0.382516
12         1704.330672  2042.698918 -0.447100

Overall mean:
MAE     1321.136271
RMSE    1593.781052
R2        -0.033

#### Multi-Horizon: TimeSeries Cross-Validation

In [ ]:
TEST_SIZE = 30 * 24
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

cv_rows_m = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_multi), start=1):
    Xtr, Xte = X_multi.iloc[train_idx], X_multi.iloc[test_idx]
    Ytr, Yte = Y_multi.iloc[train_idx], Y_multi.iloc[test_idx]

    model_multi.fit(Xtr, Ytr)
    Yp = model_multi.predict(Xte)

    for j, h in enumerate(HORIZONS):
        y_true_h = Yte.iloc[:, j].values
        y_pred_h = Yp[:, j]
        cv_rows_m.append({
            "fold":      fold,
            "horizon_h": h,
            "MAE":       mean_absolute_error(y_true_h, y_pred_h),
            "RMSE":      rmse(y_true_h, y_pred_h),
            "R2":        r2_score(y_true_h, y_pred_h),
        })

results_multi_cv = pd.DataFrame(cv_rows_m)

print("[Multi-Horizon] Per-horizon mean across folds:")
summary = results_multi_cv.groupby("horizon_h")[["MAE", "RMSE", "R2"]].mean()
print(summary)
print(f"\nOverall mean:\n{results_multi_cv[['MAE', 'RMSE', 'R2']].mean()}")

try:
    display(
        summary.style
        .background_gradient(subset=["MAE", "RMSE"], cmap="Reds_r", axis=0)
        .background_gradient(subset=["R2"],           cmap="Greens",  axis=0)
        .format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
    )
except NameError:
    pass

#### Multi-Horizon: Refit on train (up to TEST_START), save ONNX + weights

In [ ]:
# Refit on training data (everything before TEST_START) for the final saved model
print("Refitting multi-horizon model on train data (up to TEST_START)...")
model_multi.fit(X_train_m, Y_train_m)

# Save sklearn weights (joblib)
joblib_path_multi = os.path.join(MODEL_DIR, "gap_multi_horizon.joblib")
joblib.dump(model_multi, joblib_path_multi)
print(f"Saved sklearn weights: {joblib_path_multi}")

# Save ONNX
initial_type_multi = [("float_input", FloatTensorType([None, X_multi.shape[1]]))]
onnx_multi = convert_sklearn(model_multi, initial_types=initial_type_multi, target_opset=17)

onnx_path_multi = os.path.join(MODEL_DIR, "gap_multi_horizon.onnx")
with open(onnx_path_multi, "wb") as f:
    f.write(onnx_multi.SerializeToString())
print(f"Saved ONNX model: {onnx_path_multi}")

# Save feature columns list for inference
joblib.dump(feature_cols_multi, os.path.join(MODEL_DIR, "feature_cols_multi.joblib"))

# Sanity check ONNX
sess_multi = rt.InferenceSession(onnx_path_multi)
sample_m = X_test_m.iloc[:3].values.astype(np.float32)
preds_onnx_m = sess_multi.run(None, {sess_multi.get_inputs()[0].name: sample_m})
print(f"ONNX sanity check output shape: {np.array(preds_onnx_m[0]).shape}")

# Verify ONNX matches sklearn
preds_sk_m = model_multi.predict(X_test_m.iloc[:3])
print(f"Max difference sklearn vs ONNX: {np.max(np.abs(preds_sk_m - preds_onnx_m[0])):.6f}")

---
### Summary of saved artifacts

| Model | ONNX | Weights (joblib) | Feature list |
|-------|------|------------------|--------------|
| Single-horizon (1h) | `saved_models/gap_single_horizon.onnx` | `saved_models/gap_single_horizon.joblib` | `saved_models/feature_cols_single.joblib` |
| Multi-horizon (1-12h) | `saved_models/gap_multi_horizon.onnx` | `saved_models/gap_multi_horizon.joblib` | `saved_models/feature_cols_multi.joblib` |

In [60]:

# Save multi-horizon test set as parquet and npy
test_df_multi = X_test_m.copy()
test_df_multi["StartTime"] = times_multi.loc[test_mask_m].values
for j, h in enumerate(HORIZONS):
    test_df_multi[f"y_true_{h}h"] = Y_test_m.iloc[:, j].values
    test_df_multi[f"y_pred_{h}h"] = Y_pred_m[:, j]
test_df_multi.to_parquet(os.path.join(MODEL_DIR, "test_multi_horizon.parquet"), index=False)
np.save(os.path.join(MODEL_DIR, "test_multi_horizon_X.npy"), X_test_m.values)
np.save(os.path.join(MODEL_DIR, "test_multi_horizon_Y.npy"), Y_test_m.values)
print(f"Saved test_multi_horizon.parquet  shape: {test_df_multi.shape}")
print(f"Saved test_multi_horizon_X.npy    shape: {X_test_m.values.shape}")
print(f"Saved test_multi_horizon_Y.npy    shape: {Y_test_m.values.shape}")

Saved test_multi_horizon.parquet  shape: (156, 78)
Saved test_multi_horizon_X.npy    shape: (156, 53)
Saved test_multi_horizon_Y.npy    shape: (156, 12)
